# 02 – Modeling

**Goal:** Build the preprocessing pipeline, compare several models, tune
their hyperparameters and save the best pipeline.

- **Input:** `data/clean.csv` (created by `01_eda.ipynb`)
- **Output:** `models/personality_pipeline.joblib` (not committed to Git)
- **Main metric:** `f1_macro`
- **Random state:** `42` everywhere (taken from `src/config.py`)

> Run `01_eda.ipynb` first. Run this notebook from the `notebooks/`
> folder.

## 1. Setup

In [1]:
import sys
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

# The notebook lives in notebooks/, so the project root is made importable
sys.path.insert(0, str(Path.cwd().parent))

from src import config  # noqa: E402

print(f"Random state: {config.RANDOM_STATE}")

Random state: 42


## 2. Load the cleaned data and split it

In [2]:
clean_df = pd.read_csv(config.CLEAN_DATA_PATH)
print(f"Rows: {clean_df.shape[0]:,} | Columns: {clean_df.shape[1]}")

# The feature order is fixed by the config and reused by the app later
X = clean_df[config.FEATURE_COLUMNS]
y = clean_df[config.TARGET_COLUMN]

Rows: 19,514 | Columns: 23


In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=config.TEST_SIZE,
    stratify=y,  # keep the class proportions in both parts
    random_state=config.RANDOM_STATE,
)

print(f"Train: {X_train.shape[0]:,} rows | Test: {X_test.shape[0]:,} rows")

Train: 15,611 rows | Test: 3,903 rows


In [4]:
class_share = pd.DataFrame(
    {
        "full_%": y.value_counts(normalize=True) * 100,
        "train_%": y_train.value_counts(normalize=True) * 100,
        "test_%": y_test.value_counts(normalize=True) * 100,
    }
).round(2)
print(class_share)

# The stratified split must preserve the class proportions closely
assert (class_share["train_%"] - class_share["full_%"]).abs().max() < 0.1
assert (class_share["test_%"] - class_share["full_%"]).abs().max() < 0.1

                 full_%  train_%  test_%
target                                  
Moderate          42.84    42.84   42.84
Resilient         31.28    31.28   31.28
Overcontroller    14.41    14.41   14.40
Undercontroller   11.48    11.48   11.48


The data is split once into **80 % training** and **20 % test** data.
The split is **stratified**, so every class has (almost) the same share in
both parts. This is important because the classes are imbalanced.

The test set is only used for the final evaluation. Model selection and
hyperparameter tuning are done with cross-validation on the training
data.